# 7. t-Test — Comparing Two Groups' Continuous Measurements

**Building a Heart Disease Risk-Screening System — Notebook 7 of 12, Stage 4: Validating Claims Before Trusting Them**

Notebook 6 built the general hypothesis-testing framework and tested a
*proportion* (disease rate by sex). This notebook specializes that same framework
to a different question shape: **do patients with heart disease have higher
cholesterol, on average, than those without?** — comparing a *continuous
measurement* between two groups, the t-test's home turf.

## The topic

The t-test compares two groups' means. Unlike a z-test, it doesn't assume the
population standard deviation is known exactly — it estimates it from the sample,
which is why the t-distribution (heavier-tailed than Normal, especially at small
sample sizes) governs it instead.

## Why it matters for this system

If cholesterol's difference between groups survives this test, it's a candidate
input worth including in the regression model (Notebook 9) with real confidence
behind it. If it doesn't survive — or survives with a tiny effect size — that's
equally valuable information: it tells the system not to lean on this input more
than the evidence supports.

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats

df = pd.read_csv("../5. MLOps/2. End-to-End ML/data/heart_disease_cleaned_2.csv", index_col=0)
disease = df.loc[df["target"] == 1, "chol"]
no_disease = df.loc[df["target"] == 0, "chol"]
print(f"Disease group:    n={len(disease)}, mean cholesterol={disease.mean():.1f}")
print(f"No-disease group: n={len(no_disease)}, mean cholesterol={no_disease.mean():.1f}")

## The toolkit

| Variant | Use when |
|---|---|
| **Student's t-test** | Comparing two independent groups, assuming equal variances |
| **Welch's t-test** | Comparing two independent groups, *not* assuming equal variances (safer default) |
| **Paired t-test** | Comparing the *same* subjects measured twice (before/after) |
| **Mann-Whitney U** | Non-parametric fallback when normality is badly violated |

## How to choose

Check whether the two groups are the *same subjects measured twice* or genuinely
*different individuals* — that single question picks paired vs. independent, and
getting it wrong in either direction is a real, common mistake (using paired logic
on independent groups fabricates precision; using independent logic on paired data
throws away real information, shown directly below). Our registry has exactly one
measurement per patient, so every comparison here is necessarily the
**independent-samples** case — a follow-up-visit dataset would open up the paired
version instead. Between Student's and Welch's, default to Welch's unless you've
specifically checked variances are equal — it costs almost nothing when they are,
and protects you when they aren't. Reach for Mann-Whitney only if normality is
badly violated and the sample is too small for the t-test's robustness to save you.

## Applied to the registry

### Why t, not z

In [ ]:
import matplotlib.pyplot as plt
xs = np.linspace(-4, 4, 300)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(xs, stats.norm.pdf(xs), "k--", label="Normal (z)", lw=2)
for dof in [2, 10, 30]:
    ax.plot(xs, stats.t.pdf(xs, dof), label=f"t, df={dof}")
ax.legend(); ax.set_title("t has heavier tails at low df -- converges to Normal as df grows")
plt.show()

### Check assumptions before trusting the test

In [ ]:
shapiro_disease = stats.shapiro(disease)
shapiro_no_disease = stats.shapiro(no_disease)
print(f"Shapiro-Wilk, disease group:    p={shapiro_disease.pvalue:.4f}")
print(f"Shapiro-Wilk, no-disease group: p={shapiro_no_disease.pvalue:.4f}")

levene_result = stats.levene(disease, no_disease)
print(f"\nLevene's test for equal variances: p={levene_result.pvalue:.4f}")
print(f"Disease group std: {disease.std():.1f}   No-disease group std: {no_disease.std():.1f}")

### Welch's t-test — the safer default

In [ ]:
t_stat, p_value = stats.ttest_ind(disease, no_disease, equal_var=False)
print(f"Welch's t-test: t={t_stat:.3f}, p={p_value:.4f}")

t_classic, p_classic = stats.ttest_ind(disease, no_disease, equal_var=True)
print(f"Classic (equal-variance) t-test: t={t_classic:.3f}, p={p_classic:.4f}")
print(f"\nDifference in cholesterol: {disease.mean() - no_disease.mean():+.1f} mg/dL")

### Confidence interval for the difference

In [ ]:
diff = disease.mean() - no_disease.mean()
se_diff = np.sqrt(disease.var(ddof=1)/len(disease) + no_disease.var(ddof=1)/len(no_disease))
dof_welch = (disease.var(ddof=1)/len(disease) + no_disease.var(ddof=1)/len(no_disease))**2 / (
    (disease.var(ddof=1)/len(disease))**2/(len(disease)-1) + (no_disease.var(ddof=1)/len(no_disease))**2/(len(no_disease)-1)
)
ci_low, ci_high = stats.t.interval(0.95, dof_welch, loc=diff, scale=se_diff)
print(f"Difference in mean cholesterol: {diff:.1f} mg/dL")
print(f"95% CI for the difference: [{ci_low:.1f}, {ci_high:.1f}]")

### Effect size — Cohen's d

Like Notebook 6's Cohen's h for the proportion test, this answers "how big," not
just "how confident.\"

In [ ]:
pooled_std = np.sqrt(((len(disease)-1)*disease.var(ddof=1) + (len(no_disease)-1)*no_disease.var(ddof=1))
                      / (len(disease) + len(no_disease) - 2))
cohens_d = diff / pooled_std
print(f"Cohen's d = {cohens_d:.3f}")
print("0.2=small, 0.5=medium, 0.8=large -- this is the number that tells the system whether")
print("cholesterol is worth weighting heavily, independent of how small the p-value looks.")

### When assumptions genuinely fail: Mann-Whitney U

In [ ]:
u_stat, p_mw = stats.mannwhitneyu(disease, no_disease, alternative="two-sided")
print(f"Mann-Whitney U: U={u_stat:.1f}, p={p_mw:.4f}")
print(f"Welch's t-test p-value for comparison: {p_value:.4f}")
print("Agreement here is a useful cross-check, not usually the primary test.")

### Why the paired/independent distinction matters — illustrated with a wrong comparison

Even without paired data of our own, it's worth seeing *why* the distinction from
the toolkit section matters. Simulate what an independent-samples test would say if
it were mistakenly applied to data that was actually paired (same subject measured
twice) — this is illustrative only, since our registry has no repeat-measure data,
but the mechanism generalizes directly to any future follow-up-visit dataset this
system might incorporate.

In [ ]:
rng = np.random.default_rng(3)
n = 40
baseline = rng.normal(0, 15, n)  # per-subject baseline variability
true_effect = 5
followup = baseline + true_effect + rng.normal(0, 3, n)  # small measurement noise on top

t_paired, p_paired = stats.ttest_rel(baseline, followup)
t_unpaired, p_unpaired = stats.ttest_ind(baseline, followup)

print(f"Correctly paired test:   t={t_paired:.3f}, p={p_paired:.6f}")
print(f"Incorrectly treated as independent groups: t={t_unpaired:.3f}, p={p_unpaired:.6f}")
print("\nThe paired test is far more sensitive because it removes each subject's own baseline")
print("variability from the comparison -- exactly the toolkit distinction above, made concrete.")

## Systems view — what this stage hands to the next one

Cholesterol's group difference is now backed by a confidence interval and an
effect size, not just a p-value — the system can weight it accordingly. Notebook 8
completes Stage 4 by handling the remaining question shape this framework covers:
comparing *categorical* variables (like chest pain type) across groups, rather than
continuous measurements.

## Try it yourself

1. Run the same comparison on `thalach` (max heart rate) between disease groups —
   is the effect size larger or smaller than for cholesterol?
2. Split by `sex` instead of `target` and t-test whether `trestbps` differs by
   sex — check Levene's result first to decide whether Welch's correction matters
   much here.
3. In the paired-vs-independent illustration, shrink the per-subject baseline
   variability (`rng.normal(0, 15, n)` to `rng.normal(0, 3, n)`) — at what point
   does the paired test's advantage over the independent version shrink to
   negligible?